# TÜV Hessen source adapter

Control panel for `tuev_hessen.run` (`1_scrapers/tuev_hessen/src/run.py`). All logic lives in the module; this notebook only calls it. Every path comes from `core.paths`. No key, no login.

**How it works.** The locator page `tuev-hessen.de/38/alle-standorte-...` posts a form to `/?eID=location_search` and gets all sites back as Elasticsearch JSON, with no paging and no size field. The site is behind Cloudflare, so the adapter drives a real Chromium with Playwright (headless first, headed once if headless is blocked). It runs two searches through the UI, **"Alle Standorte"** and **"Kfz Prüfstelle"**, and replays each request from inside the page.

**HU site** = tagged "Kfz Prüfstelle". **Complete** = `hits.total` (`eq`) = number of hits = UI counter, and UI answer = replay answer. "Alle Standorte" only means "tagged with one of the three fixed tags" (Kfz Prüfstelle, Managementsysteme, MPU Begutachtung).

**Section order:** 2 re-parse a saved run offline, 3 inspect, 4 fetch (simulated, live), 5 compare two runs. Sections 2-3 need no network and no browser.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from core import paths
from tuev_hessen import run as hessen

ROOT = paths.project_root()
OUT = paths.output_dir("tuev_hessen", "full")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)


def live_runs():
    """Official live runs that kept their raw answers (the source of an offline rebuild), oldest first."""
    return sorted(p for p in OUT.glob("*_full") if (p / paths.RAW_FILE).exists() and (p / paths.RUN_SUMMARY_JSON).exists())


print("Live runs with raw data:", [p.name for p in live_runs()])
ROOT

## 2. Offline: re-parse a saved run (no requests, no browser)

Parses and checks the `raw.jsonl` of a live run again with the current code and writes a `rebuild` run folder in `1_scrapers/tuev_hessen/output/`. Use it after changing a parser rule. Each execution makes a new folder; nothing is overwritten. `SOURCE` defaults to the newest live run.

In [ ]:
SOURCE = live_runs()[-1]  # or a folder name: OUT / "2026-10-06_1059_full"

rb = hessen.run(mode="rebuild", source_run=SOURCE)
print(rb.status, "-", rb.summary["outcome"])
print("source:", SOURCE.name, "| new folder:", rb.run_dir.relative_to(ROOT))
print("counts:", rb.summary["counts"])
pd.DataFrame(rb.summary["checks"])[["severity", "ok", "check", "detail"]]

## 3. Inspect

In [ ]:
hu, al = rb.frames["hu"], rb.frames["all"]
print(f"HU sites: {len(hu)} | all listed: {len(al)} | non-HU: {len(al) - len(hu)}")
print()
pd.crosstab(al["site_type"], al["is_hu_site"].map({True: "HU", False: "not HU"}), margins=True)

In [ ]:
# Checks that are not OK (review = a finding for a human, info = only listed, hard = fails the run)
chk = pd.DataFrame(rb.summary["checks"])
chk[~chk["ok"]][["severity", "check", "detail"]]

In [ ]:
# Unclassified titles (anything that is not "Service-Center", "Partner" or "TÜV SÜD Auto Partner GmbH") and the non-HU sites
cols = ["uid", "title", "tags", "is_hu_site", "is_lkw_site", "hu_evidence"]
print("Unclassified:")
display(al[al["site_type"] == "unclassified"][cols])
print("Not HU (only in 'Alle Standorte'):")
al[~al["is_hu_site"]][["uid", "title", "tags", "hu_evidence"]]

In [ ]:
# Data quality (nothing is corrected, only flagged)
f = rb.summary["findings"]
print("Coordinates shared by different addresses:")
for g in f["coords_shared_with_other_address"]:
    print("  uids", g["uids"], "->", g["addresses"])
print("Far from the centroid of their PLZ (uid, km):", f["coords_far_from_plz"], "| distances:", f["plz_distance"])
print("Booking locationSign shared by several sites:", f["shared_location_signs"])
print("Address shared by several uids:", [(g["uids"], g["address"]) for g in f["address_shared"]])
print("Outside the Germany box:", f["coords_outside_germany"], "| missing coordinates:", f["coords_missing"])
print("Own stations without the 'TÜH Service-Center' tag:", len(f["tag_title_mismatch"]))
print("HU tag but no HU wording in the record:", f["hu_tag_without_hu_evidence"])
print("No phone:", f["missing_phone"])

In [ ]:
review = al[al["needs_review"]]
print(f"{len(review)} rows are flagged for review (they stay in the export)")
review[["uid", "site_type", "is_hu_site", "street", "zip", "city", "dq_flags"]]

In [ ]:
# Footprint: HU sites by Bundesland (from the PLZ, GeoNames)
from core.geo import ensure_geonames, load_plz_centroids

cent = load_plz_centroids(ensure_geonames(paths.geonames_dir())).set_index("zip")
state = hu["zip"].map(cent["state"])
print(state.value_counts(dropna=False).to_string())
print()
hu.loc[state != "Hessen", ["uid", "site_type", "zip", "city"]].assign(state=state)

In [ ]:
# Five random HU sites to look up on Google Maps (change SEED for another five)
SEED = 7
pick = hu.sample(5, random_state=SEED)[["uid", "site_type", "street", "zip", "city", "latitude", "longitude"]].copy()
pick["maps"] = "https://www.google.com/maps?q=" + pick["latitude"].astype(str) + "," + pick["longitude"].astype(str)
pick.drop(columns=["latitude", "longitude"])

In [ ]:
# How many HU sites have each field (non-empty values)
fields = ["phone", "mobile", "fax", "opening_hours", "booking_url", "location_sign", "notices", "betreiber_text", "websites", "detail_url"]
(hu[fields].ne("").sum().rename("of %d HU sites" % len(hu))).to_frame()

In [ ]:
# The two detail pages the live run opened (only what they offer; the run does not crawl them)
for p in rb.summary["detail_pages"]:
    d = p["description"]
    print(p["url"], "| HTTP", p["status"], "|", d.get("headings"), "| rating:", d.get("google_rating_text") or "-")

## 4. Fetch

Sections 4a-4b write one run folder `YYYY-MM-DD_HHMM_<mode>`. The cells are self-contained: they run right after a kernel restart without the cells above.

### 4a. Simulated run (no requests, no browser)

A fake locator with known truth (10 sites, 8 of them HU, one copy-paste coordinate error, one LKW title). Must PASS with 8 HU sites. Goes to `test_output/`.

In [ ]:
import pandas as pd
from tuev_hessen import run as hessen

sim = hessen.run(mode="simulated")
print(sim.status, "-", sim.summary["counts"], "(expected: all 10, hu 8, non_hu 2)")
sim.frames["hu"][["uid", "title", "site_type", "dq_flags"]].tail(4)

### 4b. Full run (live, 3 page loads + 4 searches, about 35 seconds)

**Live requests.** Needs `confirm_full_run=True`. The committed default is `False`: with it the cell stops with an error on purpose, before any browser is started. What happens: Chromium opens the locator page, declines the cookies, runs "Alle Standorte" and "Kfz Prüfstelle" through the UI, replays both requests from the page, and opens two detail pages. Pauses of 2-4 s, one thing at a time. Cloudflare block, error page, non-JSON or empty answer = FAIL, never "0 sites". `HEADLESS = True` starts without a window; if that is blocked, the run retries once with a visible window (`FALLBACK_HEADED`).

Expect about 131 listed sites and 113 HU sites (2026-10-06), but the figures come from the run: compare them with the summary, not with this text. A drop of more than 10 % against the last passing full run fails the run. Works inside Jupyter (the browser runs in a worker thread). First time on a machine: `.venv\Scripts\python.exe -m playwright install chromium`.

In [ ]:
from core import paths
from tuev_hessen import run as hessen

ROOT = paths.project_root()
HEADLESS = True
FALLBACK_HEADED = True

# Set confirm_full_run=True to start the full run. Keep False in the committed notebook.
full = hessen.run(mode="full", confirm_full_run=False, root=ROOT, headless=HEADLESS, fallback_headed=FALLBACK_HEADED)

In [ ]:
# After a full run: outcome, failed checks, review findings
import pandas as pd

if "full" not in globals():
    print("No full run in this session: set confirm_full_run=True in the cell above and run it first.")
else:
    print(full.status, "-", full.summary["outcome"])
    print("run folder:", full.run_dir.relative_to(ROOT))
    print("counts:", full.summary["counts"], "| drop check:", full.summary.get("drop_check"))
    if full.summary.get("error"):
        print("ERROR:", full.summary["error"])
    chk = pd.DataFrame(full.summary["checks"])
    display(chk[~chk["ok"]][["severity", "check", "detail"]] if len(chk) else chk)

## 5. Compare two runs (change detection)

Which sites appeared, disappeared or changed between two runs of `output/`. Set `OLD` and `NEW` to run-folder names (any mix of `_full` and `_rebuild`).

In [ ]:
import pandas as pd
from core import paths

out = paths.output_dir("tuev_hessen", "full")
runs = sorted(p.name for p in out.iterdir() if p.is_dir() and any(p.glob("tuev_hessen_all_*.csv")))
print("Runs in output/:", runs)

OLD, NEW = None, None  # e.g. "2026-10-06_1102_rebuild", "2026-11-03_0900_full"
if OLD and NEW:
    def load(name):
        f = sorted((out / name).glob("tuev_hessen_all_*.csv"))[0]
        return pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False).set_index("source_id")
    old, new = load(OLD), load(NEW)
    both = old.index.intersection(new.index)
    skip = {"scraped_at"}
    changed = {uid: [c for c in old.columns if c in new.columns and c not in skip and old.at[uid, c] != new.at[uid, c]] for uid in both}
    changed = {uid: cols for uid, cols in changed.items() if cols}
    print(f"{len(old)} -> {len(new)} | new: {sorted(new.index.difference(old.index), key=int)} | gone: {sorted(old.index.difference(new.index), key=int)}")
    print(f"{len(changed)} sites with changed fields")
    display(pd.Series({uid: ", ".join(cols) for uid, cols in changed.items()}, name="changed columns").to_frame())